# Phase 8: RNNs & Sequence Models

## What you'll learn
- RNN, LSTM, GRU — how they work
- Packed sequences for variable-length inputs
- Bidirectional RNNs
- Sequence-to-sequence architecture
- Text classification project

---

## Why RNNs?

RNNs process **sequential data** — text, time series, audio — where order matters.  
They maintain a **hidden state** that carries information across time steps.

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 8.1 nn.RNN — Vanilla RNN

At each time step: `h_t = tanh(W_ih * x_t + W_hh * h_{t-1} + b)`

**Problem**: Vanilla RNNs suffer from vanishing gradients on long sequences.

In [ ]:
# RNN(input_size, hidden_size, num_layers, batch_first)
rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

# Input: (batch, seq_len, input_size)
x = torch.randn(4, 15, 10)  # batch=4, seq_len=15, features=10

# output: all hidden states, h_n: last hidden state
output, h_n = rnn(x)

print(f"Input:  {x.shape}")
print(f"Output: {output.shape}")  # (4, 15, 20) — hidden state at each step
print(f"h_n:    {h_n.shape}")     # (2, 4, 20) — last hidden for each layer

## 8.2 nn.LSTM — Long Short-Term Memory

LSTMs solve the vanishing gradient problem with **gates** (forget, input, output) and a **cell state**.

This is the most commonly used RNN variant.

In [ ]:
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

x = torch.randn(4, 15, 10)

# LSTM returns (output, (h_n, c_n)) — c_n is the cell state
output, (h_n, c_n) = lstm(x)

print(f"Output: {output.shape}")  # (4, 15, 20)
print(f"h_n:    {h_n.shape}")     # (2, 4, 20)
print(f"c_n:    {c_n.shape}")     # (2, 4, 20)

# Get last time step output (common for classification)
last_output = output[:, -1, :]  # (4, 20)
print(f"Last step: {last_output.shape}")

## 8.3 nn.GRU — Gated Recurrent Unit

Simpler than LSTM (no cell state), often similar performance, faster to train.

In [ ]:
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

x = torch.randn(4, 15, 10)
output, h_n = gru(x)  # no cell state

print(f"Output: {output.shape}")  # (4, 15, 20)
print(f"h_n:    {h_n.shape}")     # (2, 4, 20)

## 8.4 Bidirectional RNNs

Process the sequence in **both directions** — captures context from past AND future.

In [ ]:
bilstm = nn.LSTM(
    input_size=10, hidden_size=20,
    num_layers=2, batch_first=True,
    bidirectional=True
)

x = torch.randn(4, 15, 10)
output, (h_n, c_n) = bilstm(x)

# Output hidden_size is doubled (forward + backward)
print(f"Output: {output.shape}")  # (4, 15, 40) — 20*2
print(f"h_n:    {h_n.shape}")     # (4, 4, 20) — 2layers*2directions

## 8.5 Packed Sequences — Variable Length Inputs

Real sequences have different lengths. Packing avoids wasting computation on padding.

In [ ]:
# Sequences of different lengths
seq1 = torch.randn(5, 10)  # length 5
seq2 = torch.randn(3, 10)  # length 3
seq3 = torch.randn(7, 10)  # length 7

# Pad to same length
padded = nn.utils.rnn.pad_sequence([seq1, seq2, seq3], batch_first=True)
lengths = torch.tensor([5, 3, 7])
print(f"Padded: {padded.shape}")  # (3, 7, 10)

# Pack (must sort by length descending)
sorted_lengths, sort_idx = lengths.sort(descending=True)
sorted_padded = padded[sort_idx]

packed = pack_padded_sequence(sorted_padded, sorted_lengths, batch_first=True)

# Run through LSTM
lstm = nn.LSTM(10, 20, batch_first=True)
packed_output, (h_n, c_n) = lstm(packed)

# Unpack
output, output_lengths = pad_packed_sequence(packed_output, batch_first=True)
print(f"Unpacked output: {output.shape}")
print(f"Lengths: {output_lengths}")

## 8.6 Text Classification with LSTM

Complete example: classify text sequences using an embedding + LSTM model.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # x: (batch, seq_len) — token indices
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        output, (h_n, _) = self.lstm(embedded)

        # Concatenate last hidden states from both directions
        # h_n shape: (num_layers*2, batch, hidden_dim)
        hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, hidden_dim*2)
        return self.fc(self.dropout(hidden))

# Test
model = TextClassifier(vocab_size=10000, embed_dim=128, hidden_dim=256, num_classes=2)
dummy_tokens = torch.randint(0, 10000, (4, 50))  # batch=4, seq_len=50
output = model(dummy_tokens)
print(f"Output: {output.shape}")  # (4, 2)

## 8.7 Seq2Seq Architecture (Encoder-Decoder)

Used for tasks where input and output are both sequences (translation, summarization).

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden, cell):
        embedded = self.embedding(x.unsqueeze(1))  # (batch, 1, embed)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, target_vocab_size):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.target_vocab_size = target_vocab_size

    def forward(self, src, trg):
        batch_size, trg_len = trg.shape
        outputs = torch.zeros(batch_size, trg_len, self.target_vocab_size).to(src.device)

        hidden, cell = self.encoder(src)
        decoder_input = trg[:, 0]  # <SOS> token

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, t] = output
            decoder_input = trg[:, t]  # teacher forcing

        return outputs

# Test
enc = Encoder(5000, 128, 256)
dec = Decoder(5000, 128, 256)
seq2seq = Seq2Seq(enc, dec, 5000)

src = torch.randint(0, 5000, (2, 10))
trg = torch.randint(0, 5000, (2, 12))
out = seq2seq(src, trg)
print(f"Seq2Seq output: {out.shape}")  # (2, 12, 5000)

## ✅ Phase 8 Summary

You now know how to:
- Use RNN, LSTM, GRU for sequence processing
- Handle variable-length sequences with packing
- Build bidirectional models for richer context
- Create text classifiers with Embedding + LSTM
- Build encoder-decoder (Seq2Seq) architectures

**Next → Phase 9: Transformers & Attention**